# RNNs and Sequence Models

Comprehensive guide to Recurrent Neural Networks (RNNs), LSTMs, GRUs, and practical sequence modeling applications.

📺 **Video Lecture:** [https://youtu.be/G3vQTk-kq9g](https://youtu.be/G3vQTk-kq9g)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {torch.device('cpu')}")

## 1. Vanilla RNN from Scratch (NumPy)

### RNN Equations:
- **Hidden state update:** $h_t = \\tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$
- **Output:** $y_t = W_{hy} h_t + b_y$

Where:
- $x_t$ = input at time step $t$
- $h_t$ = hidden state at time step $t$
- $W_{hh}$, $W_{xh}$, $W_{hy}$ = weight matrices
- $b_h$, $b_y$ = bias vectors

In [ ]:
class VanillaRNN:
    """Simple RNN implementation from scratch using NumPy."""
    
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize weights
        self.Wxh = np.random.randn(hidden_size, input_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.Why = np.random.randn(output_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        
    def forward(self, X):
        """Forward pass through the RNN.
        
        Args:
            X: sequence of shape (seq_len, input_size)
        
        Returns:
            outputs: predictions at each step
            hidden_states: all hidden states
        """
        seq_len = X.shape[0]
        h = np.zeros((self.hidden_size, 1))
        hidden_states = [h]
        outputs = []
        
        for t in range(seq_len):
            x = X[t].reshape(-1, 1)
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            hidden_states.append(h)
            outputs.append(y)
        
        return outputs, hidden_states
    
    def backward(self, X, outputs, hidden_states, targets):
        """Simple BPTT (Backpropagation Through Time)."""
        seq_len = X.shape[0]
        
        # Initialize gradients
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros_like(hidden_states[0])
        
        # Backpropagate through time
        for t in reversed(range(seq_len)):
            x = X[t].reshape(-1, 1)
            h = hidden_states[t]
            h_next = hidden_states[t + 1]
            
            # Output gradient
            dy = outputs[t] - targets[t].reshape(-1, 1)
            dWhy += dy @ h_next.T
            dby += dy
            
            # Hidden state gradient
            dh = self.Why.T @ dy + dh_next
            dh_raw = (1 - h_next ** 2) * dh
            
            dWxh += dh_raw @ x.T
            dWhh += dh_raw @ h.T
            dbh += dh_raw
            dh_next = self.Whh.T @ dh_raw
        
        # Clip gradients to prevent explosion
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)
        
        # Update parameters
        self.Wxh -= self.learning_rate * dWxh
        self.Whh -= self.learning_rate * dWhh
        self.Why -= self.learning_rate * dWhy
        self.bh -= self.learning_rate * dbh
        self.by -= self.learning_rate * dby

# Example: Simple sequence - predict next value
print("=" * 50)
print("Vanilla RNN Example: Sine Wave Prediction")
print("=" * 50)

# Generate simple sine wave data
X_train = np.sin(np.linspace(0, 4*np.pi, 100)).reshape(-1, 1)
X_test = np.sin(np.linspace(4*np.pi, 6*np.pi, 30)).reshape(-1, 1)

# Create RNN
rnn = VanillaRNN(input_size=1, hidden_size=10, output_size=1, learning_rate=0.1)

# Simple forward pass
outputs, hidden_states = rnn.forward(X_train[:10])
print(f"Input shape: {X_train[:10].shape}")
print(f"Number of hidden states: {len(hidden_states)}")
print(f"Hidden state size: {hidden_states[0].shape}")
print(f"Sample output: {outputs[0].flatten()[:3]}")
print("\\nVanilla RNN successfully implemented!")

## 2. Same RNN in PyTorch

PyTorch's `torch.nn.RNN` provides an optimized implementation with support for GPU acceleration.

In [ ]:
class RNNModel(nn.Module):
    """RNN in PyTorch for sequence-to-sequence tasks."""
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, h=None):
        """Forward pass.
        
        Args:
            x: input of shape (batch, seq_len, input_size)
            h: initial hidden state (optional)
        
        Returns:
            output: predictions
            h_n: final hidden state
        """
        if h is None:
            h = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        rnn_out, h_n = self.rnn(x, h)
        # Use last output for final prediction
        output = self.fc(rnn_out[:, -1, :])
        return output, h_n

# Example
print("\\n" + "=" * 50)
print("PyTorch RNN Example")
print("=" * 50)

model = RNNModel(input_size=1, hidden_size=10, output_size=1)

# Create dummy input
batch_size, seq_len, input_size = 4, 10, 1
x = torch.randn(batch_size, seq_len, input_size)

# Forward pass
output, h_n = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Hidden state shape: {h_n.shape}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

## 3. LSTM Architecture

### LSTM Equations:
Long Short-Term Memory (LSTM) addresses the vanishing gradient problem using gates:

**Input Gate:** $i_t = \\sigma(W_{ii} x_t + W_{hi} h_{t-1} + b_i)$

**Forget Gate:** $f_t = \\sigma(W_{if} x_t + W_{hf} h_{t-1} + b_f)$

**Cell Gate:** $\\tilde{C}_t = \\tanh(W_{ic} x_t + W_{hc} h_{t-1} + b_c)$

**Cell State:** $C_t = f_t \\odot C_{t-1} + i_t \\odot \\tilde{C}_t$

**Output Gate:** $o_t = \\sigma(W_{io} x_t + W_{ho} h_{t-1} + b_o)$

**Hidden State:** $h_t = o_t \\odot \\tanh(C_t)$

Where $\\sigma$ is sigmoid and $\\odot$ is element-wise multiplication.

### Key Components:
- **Forget Gate:** Controls what to forget from previous cell state
- **Input Gate:** Controls what new information to add
- **Cell State:** Long-term memory
- **Output Gate:** Controls what to output
- **Hidden State:** Short-term memory

In [ ]:
class LSTMModel(nn.Module):
    """LSTM-based sequence model."""
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, dropout=0.0):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, h=None, c=None):
        """Forward pass.
        
        Args:
            x: input of shape (batch, seq_len, input_size)
            h: initial hidden state (optional)
            c: initial cell state (optional)
        
        Returns:
            output: predictions
            (h_n, c_n): final hidden and cell states
        """
        if h is None:
            h = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        if c is None:
            c = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        lstm_out, (h_n, c_n) = self.lstm(x, (h, c))
        output = self.fc(lstm_out[:, -1, :])
        return output, (h_n, c_n)

print("\\n" + "=" * 50)
print("LSTM Architecture Example")
print("=" * 50)

lstm_model = LSTMModel(input_size=5, hidden_size=20, output_size=2, num_layers=2)

# Dummy input
x_lstm = torch.randn(8, 15, 5)  # batch=8, seq_len=15, input_size=5

output_lstm, (h_lstm, c_lstm) = lstm_model(x_lstm)

print(f"Input shape: {x_lstm.shape}")
print(f"Output shape: {output_lstm.shape}")
print(f"Hidden state shape: {h_lstm.shape}")
print(f"Cell state shape: {c_lstm.shape}")
print(f"Total parameters: {sum(p.numel() for p in lstm_model.parameters())}")
print("\\nLSTM has 4 gates per layer, explaining the parameter count.")

## 4. GRU (Gated Recurrent Unit)

### GRU Equations:
Simpler than LSTM with 2 gates instead of 3:

**Reset Gate:** $r_t = \\sigma(W_{ir} x_t + W_{hr} h_{t-1} + b_r)$

**Update Gate:** $z_t = \\sigma(W_{iz} x_t + W_{hz} h_{t-1} + b_z)$

**Candidate:** $\\tilde{h}_t = \\tanh(W_{ih} x_t + W_{hh} (r_t \\odot h_{t-1}) + b_h)$

**Hidden State:** $h_t = (1 - z_t) \\odot \\tilde{h}_t + z_t \\odot h_{t-1}$

### Comparison with LSTM:
| Aspect | LSTM | GRU |
|--------|------|-----|
| Gates | 3 (input, forget, output) | 2 (reset, update) |
| Memory | Cell state + hidden state | Only hidden state |
| Parameters | ~4x more | Fewer |
| Computation | Slower | Faster |
| Performance | Often better on large data | Often equal on smaller data |

In [ ]:
class GRUModel(nn.Module):
    """GRU-based sequence model."""
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, dropout=0.0):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, h=None):
        """Forward pass.
        
        Args:
            x: input of shape (batch, seq_len, input_size)
            h: initial hidden state (optional)
        
        Returns:
            output: predictions
            h_n: final hidden state
        """
        if h is None:
            h = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        gru_out, h_n = self.gru(x, h)
        output = self.fc(gru_out[:, -1, :])
        return output, h_n

print("\\n" + "=" * 50)
print("GRU vs LSTM Comparison")
print("=" * 50)

lstm_model_cmp = LSTMModel(input_size=10, hidden_size=32, output_size=5, num_layers=2)
gru_model_cmp = GRUModel(input_size=10, hidden_size=32, output_size=5, num_layers=2)

lstm_params = sum(p.numel() for p in lstm_model_cmp.parameters())
gru_params = sum(p.numel() for p in gru_model_cmp.parameters())

print(f"LSTM parameters: {lstm_params}")
print(f"GRU parameters: {gru_params}")
print(f"GRU is {lstm_params/gru_params:.2f}x smaller than LSTM")
print(f"\\nFor same hidden_size, GRU is more efficient!")

## 5. Sequence Classification

Task: Train an LSTM to classify sequences based on their sum or internal pattern.

In [ ]:
def generate_sequence_classification_data(num_samples=1000, seq_len=20, num_classes=3):
    """Generate synthetic sequences for classification.
    
    Label is determined by the sum of the sequence:
    - Class 0: sum < -5
    - Class 1: -5 <= sum <= 5
    - Class 2: sum > 5
    """
    X = np.random.randn(num_samples, seq_len, 5).astype(np.float32)
    y = []
    
    for i in range(num_samples):
        seq_sum = X[i].sum()
        if seq_sum < -5:
            y.append(0)
        elif seq_sum > 5:
            y.append(2)
        else:
            y.append(1)
    
    return torch.FloatTensor(X), torch.LongTensor(y)

print("\\n" + "=" * 50)
print("Sequence Classification with LSTM")
print("=" * 50)

# Generate data
X_class, y_class = generate_sequence_classification_data(num_samples=500, seq_len=20, num_classes=3)

# Split into train/val
train_size = int(0.8 * len(X_class))
X_train_class, X_val_class = X_class[:train_size], X_class[train_size:]
y_train_class, y_val_class = y_class[:train_size], y_class[train_size:]

print(f"Training data: {X_train_class.shape}")
print(f"Validation data: {X_val_class.shape}")
print(f"Class distribution in training: {np.bincount(y_train_class.numpy())}")

# Create model
class_lstm = LSTMModel(input_size=5, hidden_size=32, output_size=3, num_layers=2, dropout=0.3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(class_lstm.parameters(), lr=0.001)

# Training loop
epochs = 30
train_losses = []
val_accs = []

for epoch in range(epochs):
    # Train
    class_lstm.train()
    train_loss = 0
    for i in range(0, len(X_train_class), 32):
        x_batch = X_train_class[i:i+32]
        y_batch = y_train_class[i:i+32]
        
        optimizer.zero_grad()
        pred, _ = class_lstm(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(class_lstm.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    
    train_losses.append(train_loss / (len(X_train_class) // 32))
    
    # Validation
    class_lstm.eval()
    with torch.no_grad():
        val_pred, _ = class_lstm(X_val_class)
        val_acc = (val_pred.argmax(1) == y_val_class).float().mean().item()
        val_accs.append(val_acc)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_losses[-1]:.4f} | Val Acc: {val_acc:.4f}")

print(f"\\nFinal validation accuracy: {val_accs[-1]:.4f}")
print(f"Initial validation accuracy: {val_accs[0]:.4f}")

## 6. Character-Level Text Generation

Train an LSTM to learn character-level patterns and generate new text.

In [ ]:
class CharLevelLSTM(nn.Module):
    """Character-level LSTM for text generation."""
    
    def __init__(self, vocab_size, hidden_size, num_layers=2):
        super(CharLevelLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None, c=None):
        x = self.embedding(x)
        if h is None:
            h = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        if c is None:
            c = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        lstm_out, (h_n, c_n) = self.lstm(x, (h, c))
        output = self.fc(lstm_out)
        return output, (h_n, c_n)
    
    def generate(self, char_to_idx, idx_to_char, seed_text, length=100, temperature=0.8):
        """Generate text from seed."""
        self.eval()
        generated = seed_text
        
        # Convert seed to indices
        x = torch.LongTensor([[char_to_idx.get(c, 0) for c in seed_text]])
        h, c = None, None
        
        with torch.no_grad():
            for _ in range(length):
                logits, (h, c) = self.forward(x, h, c)
                # Use last character prediction
                logits = logits[0, -1, :] / temperature
                probs = torch.softmax(logits, dim=0)
                idx = torch.multinomial(probs, 1).item()
                char = idx_to_char.get(idx, '?')
                generated += char
                x = torch.LongTensor([[idx]])
        
        return generated

print("\\n" + "=" * 50)
print("Character-Level Text Generation")
print("=" * 50)

# Simple text corpus
text_corpus = """The quick brown fox jumps over the lazy dog. 
Python is a powerful programming language. 
Machine learning is transforming the world. 
Neural networks learn from data. 
LSTMs help with sequential data processing."""

# Build vocabulary
chars = sorted(set(text_corpus))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"Text length: {len(text_corpus)} characters")

# Prepare training data
seq_len = 30
X_text = []
y_text = []

for i in range(len(text_corpus) - seq_len):
    X_text.append([char_to_idx[c] for c in text_corpus[i:i+seq_len]])
    y_text.append(char_to_idx[text_corpus[i+seq_len]])

X_text = torch.LongTensor(X_text)
y_text = torch.LongTensor(y_text)

print(f"Training samples: {len(X_text)}")

# Train character-level model
char_lstm = CharLevelLSTM(vocab_size=vocab_size, hidden_size=64, num_layers=2)
char_optimizer = optim.Adam(char_lstm.parameters(), lr=0.001)
char_criterion = nn.CrossEntropyLoss()

print("\\nTraining character-level LSTM...")
for epoch in range(50):
    char_lstm.train()
    total_loss = 0
    
    for i in range(0, len(X_text) - 32, 32):
        x_batch = X_text[i:i+32]
        y_batch = y_text[i:i+32]
        
        char_optimizer.zero_grad()
        logits, _ = char_lstm(x_batch)
        loss = char_criterion(logits[:, -1, :], y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(char_lstm.parameters(), max_norm=1.0)
        char_optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 15 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss/(len(X_text)//32):.4f}")

# Generate text
print("\\nGenerated text samples:")
for seed in ["The", "Python", "Neural"]:
    generated = char_lstm.generate(char_to_idx, idx_to_char, seed, length=50, temperature=0.7)
    print(f"  Seed: '{seed}' -> '{generated}'")

## 7. Bidirectional RNNs

Process sequences in both directions to capture context from past and future.

In [ ]:
class BidirectionalLSTM(nn.Module):
    """Bidirectional LSTM for sequence labeling."""
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(BidirectionalLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # bidirectional=True processes in both directions
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, bidirectional=True)
        # Output size is 2*hidden_size due to bidirectionality
        self.fc = nn.Linear(2 * hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # lstm_out shape: (batch, seq_len, 2*hidden_size)
        output = self.fc(lstm_out)
        return output

print("\\n" + "=" * 50)
print("Bidirectional LSTM")
print("=" * 50)

# Create model
bi_lstm = BidirectionalLSTM(input_size=10, hidden_size=20, output_size=5, num_layers=2)

# Test
x_bi = torch.randn(4, 15, 10)  # batch=4, seq_len=15, input_size=10
y_bi = bi_lstm(x_bi)

print(f"Input shape: {x_bi.shape}")
print(f"Output shape: {y_bi.shape}")
print(f"\\nBidirectional LSTM output shape at each timestep: {y_bi.shape[1:]}")
print(f"Output size per timestep: 2 * hidden_size (forward + backward)")

# Parameter comparison
print(f"\\nTotal parameters: {sum(p.numel() for p in bi_lstm.parameters())}")

# Create unidirectional for comparison
uni_lstm = nn.LSTM(10, 20, 2, batch_first=True, bidirectional=False)
print(f"Unidirectional LSTM params: {sum(p.numel() for p in uni_lstm.parameters())}")
print(f"Bidirectional has ~2x parameters due to processing in both directions")

## 8. Packed Sequences

Handle variable-length sequences efficiently using `pack_padded_sequence` and `pad_packed_sequence`.

In [ ]:
print("\\n" + "=" * 50)
print("Packed Sequences for Variable-Length Data")
print("=" * 50)

# Generate variable-length sequences
print("\\nGenerating variable-length sequences...")

# Create sequences of different lengths
seqs = [torch.randn(np.random.randint(5, 15), 8) for _ in range(10)]  # 10 sequences, 8 features each
lengths = torch.tensor([seq.shape[0] for seq in seqs])

print(f"Number of sequences: {len(seqs)}")
print(f"Sequence lengths: {lengths.tolist()}")

# Pad sequences to same length
padded_seqs = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True)
print(f"Padded shape: {padded_seqs.shape}")

# Create LSTM that uses packed sequences
class PackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(PackedLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, lengths):
        # Pack padded sequence
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        
        # LSTM processes packed sequence
        packed_out, (h_n, c_n) = self.lstm(packed)
        
        # Unpack sequence
        unpacked, _ = pad_packed_sequence(packed_out, batch_first=True)
        
        # Use last valid output for each sequence
        output = self.fc(h_n.squeeze(0))
        return output

packed_lstm = PackedLSTM(input_size=8, hidden_size=16, output_size=3)

# Forward pass with variable lengths
output = packed_lstm(padded_seqs, lengths)
print(f"\\nOutput shape: {output.shape}")
print("Packed sequences are more efficient for variable-length data!")

## 9. Beam Search Decoding

Implement beam search for sequence generation to find higher-quality outputs.

In [ ]:
def beam_search_decode(model, char_to_idx, idx_to_char, seed_text, 
                       beam_width=3, max_length=50, temperature=1.0):
    """Beam search decoding for text generation.
    
    Args:
        model: CharLevelLSTM model
        char_to_idx: character to index mapping
        idx_to_char: index to character mapping
        seed_text: starting text
        beam_width: number of beams to track
        max_length: maximum generation length
        temperature: sampling temperature
    
    Returns:
        List of (text, score) tuples sorted by score
    """
    model.eval()
    
    # Initialize beams: (text, log_prob, h, c)
    x = torch.LongTensor([[char_to_idx.get(c, 0) for c in seed_text]])
    _, (h, c) = model(x)
    
    beams = [(seed_text, 0.0, h, c)]
    
    with torch.no_grad():
        for step in range(max_length):
            all_candidates = []
            
            for text, score, h, c in beams:
                # Get last character
                last_char_idx = char_to_idx.get(text[-1], 0)
                x = torch.LongTensor([[last_char_idx]])
                
                logits, (h_new, c_new) = model(x, h, c)
                logits = logits[0, -1, :] / temperature
                probs = torch.softmax(logits, dim=0)
                log_probs = torch.log(probs + 1e-10)
                
                # Get top-k candidates
                top_k_log_probs, top_k_indices = torch.topk(log_probs, beam_width)
                
                for i in range(beam_width):
                    idx = top_k_indices[i].item()
                    new_char = idx_to_char.get(idx, '?')
                    new_text = text + new_char
                    new_score = score + top_k_log_probs[i].item()
                    all_candidates.append((new_text, new_score, h_new, c_new))
            
            # Keep top beam_width candidates
            beams = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
    
    return [(text, score) for text, score, _, _ in beams]

print("\\n" + "=" * 50)
print("Beam Search Decoding")
print("=" * 50)

# Use the trained character-level LSTM
print("\\nGenerating with Beam Search (beam_width=3):")
beam_results = beam_search_decode(char_lstm, char_to_idx, idx_to_char, 
                                  seed_text="The", beam_width=3, max_length=40)

for i, (text, score) in enumerate(beam_results, 1):
    print(f"  Beam {i}: '{text}' (score: {score:.2f})")

print("\\nBeam search explores multiple paths and selects high-probability sequences!")

## 10. Visualization & Performance Analysis

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Classification training history
axes[0, 0].plot(train_losses, label='Training Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Sequence Classification: Training Loss', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Validation accuracy
axes[0, 1].plot(val_accs, label='Validation Accuracy', linewidth=2, color='green')
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Accuracy', fontsize=11)
axes[0, 1].set_title('Sequence Classification: Validation Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 1])

# Plot 3: Parameter counts
models_names = ['Vanilla RNN', 'LSTM (2 layers)', 'GRU (2 layers)', 'Bi-LSTM (2 layers)']
models_params = [
    sum(p.numel() for p in nn.RNN(10, 32, 2, batch_first=True).parameters()),
    lstm_params,
    gru_params,
    sum(p.numel() for p in nn.LSTM(10, 32, 2, batch_first=True, bidirectional=True).parameters())
]

axes[1, 0].bar(models_names, models_params, color=['skyblue', 'orange', 'green', 'red'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Number of Parameters', fontsize=11)
axes[1, 0].set_title('Parameter Count Comparison', fontsize=12, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=15)
for i, v in enumerate(models_params):
    axes[1, 0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Plot 4: Architecture comparison
arch_names = ['RNN', 'LSTM', 'GRU']
features = [1, 4, 3]  # Gates/connections
longterm = [0, 1, 0]  # Has cell state

x_pos = np.arange(len(arch_names))
width = 0.35

axes[1, 1].bar(x_pos - width/2, features, width, label='Gates/Connections', alpha=0.8, edgecolor='black')
axes[1, 1].bar(x_pos + width/2, longterm, width, label='Cell State (1=yes, 0=no)', alpha=0.8, edgecolor='black')
axes[1, 1].set_ylabel('Count', fontsize=11)
axes[1, 1].set_title('RNN Variants Comparison', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(arch_names)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/rnn_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Visualization saved!")

## Interview Takeaways

### Key Concepts:

1. **RNN Fundamentals**
   - Recurrent connections allow processing of sequences
   - Hidden state maintains information across timesteps
   - Simple but suffers from vanishing/exploding gradients

2. **LSTM Advantages**
   - 3 gates (input, forget, output) control information flow
   - Cell state acts as long-term memory
   - Addresses vanishing gradient problem
   - Best for long sequences and complex patterns

3. **GRU Benefits**
   - Simpler than LSTM (2 gates instead of 3)
   - Fewer parameters → faster training
   - Similar performance on many tasks
   - Good choice when computational efficiency matters

4. **Bidirectional Processing**
   - Captures context from both directions
   - Doubles output size and parameters
   - Essential for tasks like NER, POS tagging
   - Can't be used for real-time generation

5. **Practical Techniques**
   - **Packed sequences**: Handle variable-length inputs efficiently
   - **Gradient clipping**: Prevent exploding gradients
   - **Embedding layer**: Reduce dimensionality of discrete inputs
   - **Dropout**: Prevent overfitting in multi-layer networks
   - **Beam search**: Better generation quality than greedy decoding

### When to Use What:

| Task | Recommended | Reason |
|------|-------------|--------|
| Machine translation | LSTM/GRU | Needs long-term dependencies |
| Named entity recognition | Bi-LSTM | Needs bidirectional context |
| Text classification | LSTM + attention | Can use last hidden state |
| Time series forecasting | LSTM/GRU | Captures temporal patterns |
| Sentiment analysis | Bi-LSTM | Benefits from bidirectional context |
| Real-time speech recognition | Unidirectional LSTM | Can't wait for future data |

### Common Pitfalls:

1. **Not clipping gradients** → Exploding gradients
2. **Using RNN for very long sequences** → Vanishing gradients
3. **Not using bidirectional for classification** → Missing context
4. **Forgetting to detach hidden state** → Memory leaks
5. **Greedy decoding** → Suboptimal generation (use beam search)
6. **Not handling variable lengths** → Wasted computation on padding

### Modern Alternatives:

- **Transformer/Attention**: Better for long sequences, parallelizable
- **GRU**: When speed matters more than accuracy
- **Attention mechanism**: Can replace bidirectionality in some cases
- **Layer normalization**: Better than batch norm for RNNs

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>